In [ ]:
import os
from copy import deepcopy
import torch
random_seed = 2025
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # Replace "0" with the desired GPU device index

In [ ]:
def add_gaussian_noise(X, severity=5):
    scale = [.08, .12, 0.18, 0.26, 0.38][severity - 1]*10
    # Add Gaussian noise to the data
    noise = torch.normal(size=X.shape, std=scale, mean=0.0)
    noisy_X = X + noise
    return noisy_X

In [ ]:
class Tent:
    """Tent adapts a model by entropy minimization during testing.
    Once tented, a model adapts itself by updating on every forward.
    """
    def __init__(self, cfg, model, num_classes):
        super().__init__(cfg, model, num_classes)

        # setup loss function
        self.softmax_entropy = Entropy()

    def loss_calculation(self, x):
        imgs_test = x[0]
        outputs = self.model(imgs_test)
        loss = self.softmax_entropy(outputs).mean(0)
        return outputs, loss

    @torch.enable_grad()
    def forward_and_adapt(self, x):
        """Forward and adapt model on batch of data.
        Measure entropy of the model prediction, take gradients, and update params.
        """
        if self.mixed_precision and self.device == "cuda":
            with torch.cuda.amp.autocast():
                outputs, loss = self.loss_calculation(x)
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad()
        else:
            outputs, loss = self.loss_calculation(x)
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()
        return outputs

    def collect_params(self):
        """Collect the affine scale + shift parameters from batch norms.

        Walk the model's modules and collect all batch normalization parameters.
        Return the parameters and their names.

        Note: other choices of parameterization are possible!
        """
        params = []
        names = []
        for nm, m in self.model.named_modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.LayerNorm, nn.GroupNorm)):
                for np, p in m.named_parameters():
                    if np in ['weight', 'bias']:  # weight is scale, bias is shift
                        params.append(p)
                        names.append(f"{nm}.{np}")
        return params, names

    def configure_model(self):
        """Configure model for use with tent."""
        # train mode, because tent optimizes the model to minimize entropy
        # self.model.train()
        self.model.eval()  # eval mode to avoid stochastic depth in swin. test-time normalization is still applied
        # disable grad, to (re-)enable only what tent updates
        self.model.requires_grad_(False)
        # configure norm for tent updates: enable grad + force batch statisics
        for m in self.model.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.requires_grad_(True)
                # force use of batch stats in train and eval modes
                m.track_running_stats = False
                m.running_mean = None
                m.running_var = None
            elif isinstance(m, nn.BatchNorm1d):
                m.train()   # always forcing train mode in bn1d will cause problems for single sample tta
                m.requires_grad_(True)
            elif isinstance(m, (nn.LayerNorm, nn.GroupNorm)):
                m.requires_grad_(True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
from sklearn.model_selection import train_test_split



class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, num_classes=3):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x, return_features=False):
        x = F.relu(self.bn1(self.fc1(x)))
        features = F.relu(self.bn2(self.fc2(x)))
        logits = self.fc3(features)
        if return_features:
            return logits, features
        return logits

In [ ]:
class PointDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# 1. Create a synthetic dataset with 3 classes
n_samples = 1000
num_classes = 3

X, y = make_blobs(n_samples=n_samples, centers=num_classes, n_features=2, random_state=random_seed)

x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=random_seed)
x_train, x_val = torch.tensor(x_train, dtype=torch.float32), torch.tensor(x_val, dtype=torch.float32)
y_train, y_val = torch.tensor(y_train, dtype=torch.long), torch.tensor(y_val, dtype=torch.long)


In [ ]:

model = MLP(num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 3. Train the model on the clean data
train_losses = []
val_losses = []
n_epochs = 100
model.train()

for epoch in range(n_epochs):
    # Training
    model.train()
    optimizer.zero_grad()
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    train_loss = loss.item()
    train_losses.append(train_loss)
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(x_val)
        val_loss = criterion(val_outputs, y_val).item()
        val_losses.append(val_loss)
        
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

plt.figure(figsize=(6,3))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
# 6. Visualize the features before and after adaptation using PCA
def plot_features(features, labels, title, pca=None):
    features_np = features.detach().cpu().numpy()

    if pca is None:
        pca = PCA(n_components=2)
        features_2d = pca.fit_transform(features_np)
    else:
        features_2d = pca.transform(features_np)
    plt.figure(figsize=(6,5))
    scatter = plt.scatter(features_2d[:,0], features_2d[:,1], c=labels, cmap='viridis', alpha=0.7)
    plt.legend(*scatter.legend_elements(), title="Classes")
    plt.title(title)
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.show()

    return pca

In [ ]:
model.eval()
with torch.no_grad():
    logits, clean_features = model(x_val, return_features=True)
    accuracy = (logits.argmax(1) == y_val).float().mean().item()
    print(f"Accuracy on clean data: {accuracy:.2f}")

pca = plot_features(clean_features, y_val, "Clean Features")

In [ ]:
x_val_noisy = add_gaussian_noise(x_val, severity=5)
with torch.no_grad():
    logits, noisy_features = model(x_val_noisy, return_features=True)
    accuracy = (logits.argmax(1) == y_val).float().mean().item()
    print(f"Accuracy on noisy data: {accuracy:.2f}")

plot_features(noisy_features, y_val, "Noisy Features", pca=pca)

In [ ]:
# 5. TENT adaptation: update only BatchNorm layers using entropy minimization
def entropy_loss(logits):
    # Softmax probabilities
    probs = F.softmax(logits, dim=1)
    return -torch.mean(torch.sum(probs * torch.log(probs + 1e-6), dim=1))


def validate(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x)
        accuracy = (logits.argmax(1) == y).float().mean().item()
    return accuracy

def collect_params(model):
    """Collect the affine scale + shift parameters from batch norms.
    Walk the model's modules and collect all batch normalization parameters.
    Return the parameters and their names.
    Note: other choices of parameterization are possible!
    """
    params = []
    names = []
    for nm, m in model.named_modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.LayerNorm, nn.GroupNorm)):
            for np, p in m.named_parameters():
                if np in ['weight', 'bias']:  # weight is scale, bias is shift
                    params.append(p)
                    names.append(f"{nm}.{np}")
    return params, names

def configure_model(model):
    """Configure model for use with eata."""
    # train mode, because eata optimizes the model to minimize entropy
    # self.model.train()
    model.eval()  # eval mode to avoid stochastic depth in swin. test-time normalization is still applied
    # disable grad, to (re-)enable only what eata updates
    model.requires_grad_(False)
    # configure norm for eata updates: enable grad + force batch statisics
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.requires_grad_(True)
            # force use of batch stats in train and eval modes
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
        elif isinstance(m, nn.BatchNorm1d):
            m.train()   # always forcing train mode in bn1d will cause problems for single sample tta
            m.requires_grad_(True)
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm)):
            m.requires_grad_(True)

configure_model(model)
bn_params, bn_names = collect_params(model)
print("BatchNorm Parameters:", bn_names)
tent_optimizer = optim.Adam(bn_params, lr=1e-3)


adaptation_data = torch.cat((x_val_noisy[y_val == 0], x_val_noisy[y_val == 1]), dim=0)
# Run a few adaptation steps on the noisy data
n_adapt_steps = 300
for step in range(n_adapt_steps):
    tent_optimizer.zero_grad()
    logits = model(adaptation_data)
    loss = entropy_loss(logits)
    loss.backward()
    tent_optimizer.step()
    if (step+1) % 10 == 0:
        # test accuracy
        test_accuracy = validate(deepcopy(model), x_val_noisy, y_val)
        print(f"Adaptation step {step+1}/{n_adapt_steps}, Entropy Loss: {loss.item():.4f}")
        print(f"Test Accuracy: {test_accuracy:.2f}")


# Extract features after TENT adaptation
model.eval()
with torch.no_grad():
    logits_noisy, features_after = model(x_val_noisy, return_features=True)

In [ ]:
plot_features(features_after, y_val, "Features after TENT adaptation", pca=pca)